
# Laboratorio 2 (adaptado al español): Crear agentes para investigar y redactar un artículo con CrewAI

Este notebook está **listo para ejecutarse en Google Colab** y adapta el laboratorio original a un ejemplo completamente en español.

## Objetivos de aprendizaje
Al finalizar este laboratorio podrás:

- Definir **agentes** con `role`, `goal` y `backstory`.
- Definir **tareas** con `description` y `expected_output`.
- Construir un **crew** secuencial con CrewAI.
- Ejecutar un flujo simple de:
  1. **Planificación**
  2. **Redacción**
  3. **Edición**

## Caso de estudio
Vamos a construir un pequeño sistema multiagente que genere un artículo en español sobre un tema dado, por ejemplo:

> **Sistemas multiagente en inteligencia artificial**

---
**Sugerencia docente:** este laboratorio es ideal para introducir:
- role-playing,
- descomposición de tareas,
- coordinación secuencial,
- y prompt engineering estructurado.



## 1. Instalación de librerías

En Colab, ejecuta esta celda para instalar las dependencias necesarias.


In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29


## 2. Configuración inicial

Configuración de API_KEY y MODELO en OpenAI



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os

os.environ["OPENAI_API_KEY"] = 'sk-xxx'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'


## 3. Importaciones

Importamos las clases principales de CrewAI:

- `Agent`
- `Task`
- `Crew`


In [ ]:

from crewai import Agent, Task, Crew



## 4. Creación de agentes

Vamos a definir tres agentes:

1. **Planificador de contenido**
2. **Redactor profesional**
3. **Editor de contenido**

> En este laboratorio, los agentes trabajan de manera **secuencial**.


### 4.1 Agente: Planificador de contenido

In [ ]:

planificador = Agent(
    role="Planificador de Contenido",
    goal="Diseñar un plan atractivo, claro y bien fundamentado sobre el tema: {tema}",
    backstory=(
        "Eres un especialista en investigación y planificación de contenido. "
        "Tu trabajo consiste en analizar el tema {tema}, identificar ideas clave, "
        "estructurar un esquema sólido del artículo y orientar la redacción posterior. "
        "Debes ayudar a que la audiencia comprenda el tema y pueda tomar decisiones informadas."
    ),
    allow_delegation=False,
    verbose=True
)


### 4.2 Agente: Redactor profesional

In [ ]:

redactor = Agent(
    role="Redactor Profesional",
    goal="Escribir un artículo claro, coherente y bien estructurado sobre el tema: {tema}",
    backstory=(
        "Eres un redactor experto en divulgación técnica y académica. "
        "Te apoyas en el trabajo del planificador para redactar un artículo completo, "
        "bien argumentado y fácil de leer. "
        "Debes diferenciar claramente entre hechos, interpretaciones y opiniones."
    ),
    allow_delegation=False,
    verbose=True
)


### 4.3 Agente: Editor de contenido

In [ ]:

editor = Agent(
    role="Editor de Contenido",
    goal="Revisar y mejorar la calidad final del artículo en español",
    backstory=(
        "Eres un editor profesional con experiencia en publicaciones técnicas y académicas. "
        "Tu tarea es revisar el artículo generado por el redactor, corregir errores, "
        "mejorar claridad, cohesión y tono, y asegurar que el texto final esté listo para publicarse."
    ),
    allow_delegation=False,
    verbose=True
)



## 5. Creación de tareas

Cada tarea debe incluir al menos:

- `description`
- `expected_output`
- `agent`

Esto ayuda a que CrewAI genere mejores instrucciones internas para cada agente.


### 5.1 Tarea: Planificación

In [ ]:

tarea_planificacion = Task(
    description=(
        "1. Identifica tendencias, conceptos clave y aspectos relevantes del tema {tema}.\n"
        "2. Define la audiencia objetivo y sus principales intereses o necesidades.\n"
        "3. Elabora una estructura detallada del artículo que incluya:\n"
        "   - Introducción\n"
        "   - Secciones principales\n"
        "   - Conclusión\n"
        "4. Incluye palabras clave útiles y posibles enfoques para enriquecer el contenido."
    ),
    expected_output=(
        "Un plan de contenido completo en español, con esquema del artículo, "
        "análisis de audiencia y palabras clave relevantes."
    ),
    agent=planificador
)


### 5.2 Tarea: Redacción

In [ ]:

tarea_redaccion = Task(
    description=(
        "1. Usa el plan generado previamente para redactar un artículo sobre {tema}.\n"
        "2. Escribe en español con claridad, precisión y buena organización.\n"
        "3. Incluye subtítulos claros y atractivos.\n"
        "4. Asegura que el artículo tenga introducción, desarrollo y conclusión.\n"
        "5. Mantén un tono profesional, pedagógico y bien argumentado."
    ),
    expected_output=(
        "Un artículo completo en español, en formato Markdown, listo para revisión editorial. "
        "Cada sección debe tener entre 2 y 3 párrafos."
    ),
    agent=redactor
)


### 5.3 Tarea: Edición

In [ ]:

tarea_edicion = Task(
    description=(
        "Revisa el artículo generado y mejora su calidad final.\n"
        "- Corrige errores gramaticales y de estilo.\n"
        "- Mejora claridad, coherencia y fluidez.\n"
        "- Verifica que el tono sea profesional y consistente.\n"
        "- Mantén el contenido en formato Markdown."
    ),
    expected_output=(
        "Un artículo final en español, bien escrito, corregido y listo para publicación, "
        "en formato Markdown."
    ),
    agent=editor
)



## 6. Creación del crew

En este laboratorio, las tareas se ejecutan **en secuencia**:

1. El planificador produce el esquema.
2. El redactor escribe el artículo.
3. El editor revisa la versión final.

> El orden de las tareas importa.


In [ ]:

equipo = Crew(
    agents=[planificador, redactor, editor],
    tasks=[tarea_planificacion, tarea_redaccion, tarea_edicion],
    verbose=2
)



## 7. Ejecución del crew

Puedes cambiar el valor de `tema` por cualquier asunto de tu interés.


In [ ]:

tema = "Sistemas multiagente en inteligencia artificial"

resultado = equipo.kickoff(inputs={"tema": tema})


## 8. Visualización del resultado

In [ ]:

from IPython.display import Markdown, display

display(Markdown(resultado))



## 9. Pruébalo con otro tema

Ejemplos sugeridos:

- **Agentes IA en educación superior**
- **RAG vs Fine-Tuning**
- **Bases de datos distribuidas y escalabilidad**
- **MLOps en entornos empresariales**


In [ ]:

nuevo_tema = "Agentes IA en educación superior"

resultado_2 = equipo.kickoff(inputs={"tema": nuevo_tema})
display(Markdown(resultado_2))



## 10. Extensiones sugeridas para estudiantes

### Extensión 1
Agregar un **cuarto agente** que actúe como:
- investigador web,
- verificador de fuentes,
- o curador de referencias.

### Extensión 2
Transformar el flujo secuencial en un flujo:
- **jerárquico**, con un agente manager,
- o **paralelo**, donde dos agentes produzcan salidas complementarias.

### Extensión 3
Modificar el sistema para que genere:
- un artículo,
- una presentación breve,
- y un resumen ejecutivo del mismo tema.



## 11. Reflexión final

Este laboratorio ilustra muy bien cinco ideas centrales de los sistemas multiagente:

1. **Role-playing**: cada agente tiene una función específica.  
2. **Task decomposition**: el problema se divide en subtareas.  
3. **Workflow secuencial**: la salida de un agente alimenta al siguiente.  
4. **Prompt engineering estructurado**: `role + goal + backstory`.  
5. **Reutilización**: cambiando solo el `tema`, el mismo workflow puede reutilizarse.

---

### Próximo paso recomendado
Como evolución natural, este laboratorio puede extenderse con:
- **tools reales**,
- **búsqueda web**,
- **RAG**,
- **memoria**,
- y **procesos jerárquicos**.
